In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def procesar_dataops_limpieza():
    print("====================================================================")
    print("      PIPELINE DATAOPS: REMOCIÓN DE DUPLICADOS DE HISTORIZADOR      ")
    print("====================================================================\n")
    
    # --------------------------------------------------------------------
    # BLOQUE 1: PROCESAMIENTO DE INTENSIDADES (COURIER ALTA FRECUENCIA)
    # --------------------------------------------------------------------
    print("[PROCESO] Iniciando limpieza de 'intensidades_raw.csv'...")
    df_int = pd.read_csv('data/raw/intensidades_raw.csv')
    
    # Asegurar nombres de columnas estándar (evitando caracteres fantasma al inicio)
    df_int.columns = ['timestamp', 'n2cu', 'n1fe', 'n4mo', 'n6sc', 'n3zn']
    
    # Saneamiento temporal y ordenamiento cronológico
    df_int['timestamp'] = pd.to_datetime(df_int['timestamp'])
    df_int = df_int.sort_values('timestamp').reset_index(drop=True)
    
    total_int_raw = len(df_int)
    
    # Vector de atributos instrumentales para evaluar transiciones netas
    features_spectrometric = ['n1fe', 'n2cu', 'n3zn', 'n4mo', 'n6sc']
    
    # Conservar únicamente la primera aparición del cambio físico del Courier
    df_int_clean = df_int.drop_duplicates(subset=features_spectrometric, keep='first').reset_index(drop=True)
    total_int_clean = len(df_int_clean)
    
    # Guardar resultados limpios del Courier
    df_int_clean.to_csv('data/processed/intensidades_clean_no_duplicates.csv', index=False)
    print("[OK] 'intensidades_raw.csv' procesado y guardado.")
    
    # --------------------------------------------------------------------
    # BLOQUE 2: PROCESAMIENTO DE ASSAYS (LABORATORIO QUÍMICO CON MES EN ESPAÑOL)
    # --------------------------------------------------------------------
    print("[PROCESO] Iniciando limpieza de 'assays_raw.csv'...")
    df_ass = pd.read_csv('data/raw/assays_raw.csv')
    
    # Estandarizar nombres de columnas
    df_ass.columns = ['timestamp', 'pCu', 'pMo', 'pFe', 'pIns']
    
    total_ass_raw = len(df_ass)
    
    # --- SOLUCIÓN AL DATEPARSEERROR: Traducción explícita de meses ---
    # Diccionario para convertir abreviaturas en español al estándar que Pandas entiende
    meses_es_en = {
        '-ene-': '-Jan-', '-feb-': '-Feb-', '-mar-': '-Mar-', '-abr-': '-Apr-',
        '-may-': '-May-', '-jun-': '-Jun-', '-jul-': '-Jul-', '-ago-': '-Aug-',
        '-sep-': '-Sep-', '-oct-': '-Oct-', '-nov-': '-Nov-', '-dic-': '-Dec-'
    }
    
    # Convertir la columna a string, pasarla a minúsculas y reemplazar los meses
    df_ass['timestamp_str'] = df_ass['timestamp'].astype(str).str.lower()
    for es, en in meses_es_en.items():
        df_ass['timestamp_str'] = df_ass['timestamp_str'].str.replace(es, en)
        
    # Ahora que está en formato internacional ('01-aug-25 00:00:00'), Pandas lo parseará sin errores
    # 'mixed' ayuda si hay variaciones menores en la estructura horaria
    df_ass['timestamp'] = pd.to_datetime(df_ass['timestamp_str'], format='mixed')
    
    # Eliminar la columna auxiliar utilizada para la traducción
    df_ass = df_ass.drop(columns=['timestamp_str'])
    
    # Ordenar cronológicamente de forma estricta
    df_ass = df_ass.sort_values('timestamp').reset_index(drop=True)
    
    # Vector químico analítico para evaluar compósitos reales
    targets_chemical = ['pCu', 'pMo', 'pFe', 'pIns']
    
    # Conservar únicamente el reporte de entrada original de cada compósito de 12 horas
    df_ass_clean = df_ass.drop_duplicates(subset=targets_chemical, keep='first').reset_index(drop=True)
    total_ass_clean = len(df_ass_clean)
    
    # Guardar resultados limpios del Laboratorio
    df_ass_clean.to_csv('data/processed/assays_clean_no_duplicates.csv', index=False)
    
    # --------------------------------------------------------------------
    # REPORTE ESTADÍSTICO CAPÍTULO III (METODOLOGÍA DE INGENIERÍA DE DATA)
    # --------------------------------------------------------------------
    print("\n====================================================================")
    print("          CUADRO DE BALANCE ESTADÍSTICO PARA REDACCIÓN              ")
    print("====================================================================")
    print(f"1. DATASET DE INTENSIDADES (COURIER):")
    print(f"   - Registros Crudos SCADA:         {total_int_raw}")
    print(f"   - Muestras Únicas Reales:         {total_int_clean}")
    print(f"   - Redundancias Purgadas:          {total_int_raw - total_int_clean}")
    print(f"   - Eficiencia de Compresión:       {((total_int_raw - total_int_clean)/total_int_raw)*100:.2f}%")
    print("-" * 68)
    print(f"2. DATASET DE ENSAYOS (LABORATORIO QUÍMICO):")
    print(f"   - Registros Crudos SCADA:         {total_ass_raw}")
    print(f"   - Reportes Únicos Reales:         {total_ass_clean}")
    print(f"   - Redundancias Purgadas:          {total_ass_raw - total_ass_clean}")
    print(f"   - Eficiencia de Compresión:       {((total_ass_raw - total_ass_clean)/total_ass_raw)*100:.2f}%")
    print("====================================================================")

# Ejecutar el proceso corregido
procesar_dataops_limpieza()

      PIPELINE DATAOPS: REMOCIÓN DE DUPLICADOS DE HISTORIZADOR      

[PROCESO] Iniciando limpieza de 'intensidades_raw.csv'...
[OK] 'intensidades_raw.csv' procesado y guardado.
[PROCESO] Iniciando limpieza de 'assays_raw.csv'...

          CUADRO DE BALANCE ESTADÍSTICO PARA REDACCIÓN              
1. DATASET DE INTENSIDADES (COURIER):
   - Registros Crudos SCADA:         452161
   - Muestras Únicas Reales:         18124
   - Redundancias Purgadas:          434037
   - Eficiencia de Compresión:       95.99%
--------------------------------------------------------------------
2. DATASET DE ENSAYOS (LABORATORIO QUÍMICO):
   - Registros Crudos SCADA:         1271
   - Reportes Únicos Reales:         611
   - Redundancias Purgadas:          660
   - Eficiencia de Compresión:       51.93%
